In [70]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Remove double spaces from context
df_train['context'] = (
    df_train['context']
    .str.replace(r'\(\s*\)', '', regex=True)   # remove empty ()
    .str.replace(' +', ' ', regex=True)        # normalize spaces
    .str.strip()                               # trim leading/trailing spaces
)

# Only keep Arabic, Telugu and Korean examples
df_train = df_train[df_train['lang'].isin(['ar', 'te', 'ko'])]


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CoolD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\CoolD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [71]:
print(df_train.iloc[0]['context'])

The conflict between France and Spain continued in Catalonia until 1659, with the confrontation between two sovereigns and two Catalan governments, one based in Barcelona, under the control of Spain and the other in Perpinyà, under the occupation of France. In 1652 the French authorities renounced to Catalonia's territories south of the Pyrenees, but held control of Roussillon, thereby leading to the signing of the Treaty of the Pyrenees in 1659, which finally ended the war between France and Spain, with the partition of restive Catalonia between both empires. The Portuguese Restoration War ended with the Treaty of Lisbon in 1668, that terminated the 60-year Iberian Union.


In [72]:
## Remove unwanted characters from the questions

def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

In [73]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-multilingual-cased")

def create_bio_labels_from_offsets(context, answer_start, answer, answerable):
    """
    Convert context + answer_start/answer into token IDs + BIO labels
    using the tokenizer's offset_mapping.
    Robust to slight mismatches in character offsets.
    """
    if not answerable or answer_start == -1 or not answer:
        answerable = False
        answer_start = 0
        answer = ""

    tokens = tokenizer(context, add_special_tokens=False, return_offsets_mapping=True)
    input_ids = tokens["input_ids"]
    offsets = tokens["offset_mapping"]

    labels = ["O"] * len(input_ids)

    if answerable:
        answer_end = answer_start + len(answer)
        token_start, token_end = None, None

        # find all tokens that overlap with the answer span
        for i, (s, e) in enumerate(offsets):
            # allow small tolerance (1 char) for mismatch
            if not (e <= answer_start or s >= answer_end):
                if token_start is None:
                    token_start = i
                token_end = i

        # fallback: try nearest token if overlap not found
        if token_start is None:
            # find first token that starts after answer_start
            for i, (s, e) in enumerate(offsets):
                if s >= answer_start:
                    token_start = i
                    break
        if token_end is None:
            # find last token that ends before answer_end
            for i, (s, e) in reversed(list(enumerate(offsets))):
                if e <= answer_end:
                    token_end = i
                    break

        # last resort: just use first/last token
        if token_start is None:
            token_start = 0
        if token_end is None:
            token_end = len(input_ids) - 1

        # assign BIO labels
        labels[token_start] = "B-ANS"
        for i in range(token_start + 1, token_end + 1):
            labels[i] = "I-ANS"

    return input_ids, labels



In [74]:
row = df_train.iloc[4]

input_ids, labels = create_bio_labels_from_offsets(
    row["context"],
    row["answer_start"],
    row["answer"],
    row["answerable"]
)

tokens = tokenizer.convert_ids_to_tokens(input_ids)
for t, l in zip(tokens, labels):
    print(f"{t:15} -> {l}")

print(df_train.iloc[4])


Palestine       -> O
'               -> O
officially      -> O
the             -> O
State           -> O
of              -> O
Palestine       -> O
'               -> O
is              -> O
a               -> O
"               -> O
de              -> O
jure            -> O
"               -> O
so              -> O
##vereign       -> O
state           -> O
in              -> O
Western         -> O
Asia            -> O
claiming        -> O
the             -> O
West            -> O
Bank            -> O
border          -> O
##ing           -> O
Israel          -> O
and             -> O
Jordan          -> O
and             -> O
Gaza            -> O
Strip           -> O
border          -> O
##ing           -> O
Israel          -> O
and             -> O
Egypt           -> O
with            -> O
Jerusalem       -> O
as              -> B-ANS
the             -> I-ANS
designated      -> I-ANS
capital         -> O
although        -> O
its             -> O
administrative  -> O
center          -> O
i

In [75]:
def evaluate_bio_predictions_exact_with_examples(df, tokenizer, create_labels_fn, max_examples=10):
    """
    Count how many answers are correctly predicted in a dataset.
    Print a few examples of predictions vs ground truth.
    Returns: number_correct, total, accuracy
    """
    correct = 0
    total = 0
    printed_examples = 0

    for idx, row in df.iterrows():
        input_ids, labels = create_labels_fn(
            row["context"],
            row["answer_start"],
            row["answer"],
            row["answerable"]
        )
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        if row["answerable"]:
            total += 1

            # Reconstruct predicted answer from tokens using BIO labels
            predicted_tokens = []
            for token, label in zip(tokens, labels):
                if label in ["B-ANS", "I-ANS"]:
                    if token.startswith("##") and predicted_tokens:
                        predicted_tokens[-1] += token[2:]
                    else:
                        predicted_tokens.append(token)
            predicted_answer = " ".join(predicted_tokens)

            # normalize spaces for comparison
            predicted_answer_norm = predicted_answer.replace(" ", "").lower()
            actual_answer_norm = row["answer"].replace(" ", "").lower()

            if predicted_answer_norm == actual_answer_norm:
                correct += 1

            # Print example if we haven’t printed enough yet
            if printed_examples < max_examples:
                print(f"Context snippet: {row['context'][:80]}...")
                print(f"Ground truth answer: {row['answer']}")
                print(f"Predicted answer   : {predicted_answer}")
                print(f"Match: {predicted_answer_norm == actual_answer_norm}")
                print("-" * 50)
                printed_examples += 1

    accuracy = correct / total if total > 0 else 0
    return correct, total, accuracy

correct, total, accuracy = evaluate_bio_predictions_exact_with_examples(
    df_train,
    tokenizer,
    create_bio_labels_from_offsets,
    max_examples=10
)

print(f"Exact match answers: {correct}/{total} ({accuracy:.2%})")


Token indices sequence length is longer than the specified maximum sequence length for this model (636 > 512). Running this sequence through the model will result in indexing errors


Context snippet: The conflict between France and Spain continued in Catalonia until 1659 with the...
Ground truth answer: France
Predicted answer   : France
Match: True
--------------------------------------------------
Context snippet: X-rays make up X-radiation a form of electromagnetic radiation. Most X-rays have...
Ground truth answer: Wilhelm Röntgen
Predicted answer   : Röntgen who discovered
Match: False
--------------------------------------------------
Context snippet: In 2022 Beijing will become the first-ever city that has held both the summer an...
Ground truth answer: 2004
Predicted answer   : 2004 Summer
Match: False
--------------------------------------------------
Context snippet: The British Broadcasting Corporation BBC is a British public service broadcaster...
Ground truth answer: British Broadcasting Corporation (BBC)
Predicted answer   : British Broadcasting Corporation BBC is
Match: False
--------------------------------------------------
Context snippet: Palesti

In [76]:
def filter_correct_bio(df, tokenizer, create_labels_fn):
    """
    Return a filtered dataframe containing only rows where
    the BIO labels match the actual answer.
    """
    keep_rows = []

    for idx, row in df.iterrows():
        if not row["answerable"]:
            continue  # spring ubesvarlige over

        input_ids, labels = create_labels_fn(
            row["context"],
            row["answer_start"],
            row["answer"],
            row["answerable"]
        )
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        # Reconstruct predicted answer from BIO labels
        predicted_tokens = []
        for token, label in zip(tokens, labels):
            if label in ["B-ANS", "I-ANS"]:
                if token.startswith("##") and predicted_tokens:
                    predicted_tokens[-1] += token[2:]
                else:
                    predicted_tokens.append(token)
        predicted_answer = " ".join(predicted_tokens)

        # Normaliser for sammenligning
        predicted_answer_norm = predicted_answer.replace(" ", "").lower()
        actual_answer_norm = row["answer"].replace(" ", "").lower()

        if predicted_answer_norm == actual_answer_norm:
            keep_rows.append(True)
        else:
            keep_rows.append(False)

    # Filtrér dataframe
    filtered_df = df.iloc[[i for i, keep in enumerate(keep_rows) if keep]].reset_index(drop=True)
    return filtered_df

# Brug funktionen
df_train_correct_bio = filter_correct_bio(df_train, tokenizer, create_bio_labels_from_offsets)
df_val_correct_bio   = filter_correct_bio(df_val, tokenizer, create_bio_labels_from_offsets)

print(f"Original train size: {len(df_train)}, filtered: {len(df_train_correct_bio)}")
print(f"Original val size  : {len(df_val)}, filtered: {len(df_val_correct_bio)}")




Original train size: 6335, filtered: 2571
Original val size  : 3011, filtered: 1007


In [77]:
def simple_em_f1(pred_answers, true_answers):
    em, f1 = 0, 0
    for p, t in zip(pred_answers, true_answers):
        p_norm = p.replace(" ", "").lower()
        t_norm = t.replace(" ", "").lower()
        em += int(p_norm == t_norm)
        # F1 simplificeret: token overlap
        p_tokens, t_tokens = p_norm.split(), t_norm.split()
        common = set(p_tokens) & set(t_tokens)
        if len(common) == 0:
            f1 += 0
        else:
            f1 += 2*len(common)/(len(p_tokens)+len(t_tokens))
    n = len(pred_answers)
    return {"exact_match": 100*em/n, "f1": 100*f1/n}


In [78]:
import torch
from transformers import DistilBertTokenizerFast, AutoModelForQuestionAnswering, Trainer, TrainingArguments
from torch.utils.data import Dataset

# 0️⃣ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1️⃣ Load tokenizer og model
model_name = "distilbert-base-multilingual-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)

# 2️⃣ Dataset class
class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = row["question"]
        context = row["context"]
        answerable = row["answerable"]
        answer_text = row["answer"] if answerable else ""
        answer_start_char = row["answer_start"] if answerable else 0

        tokens = self.tokenizer(
            question,
            context,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            padding="max_length"
        )

        start_positions = torch.tensor(0)
        end_positions = torch.tensor(0)
        if answerable and answer_text:
            answer_end_char = answer_start_char + len(answer_text)
            token_start, token_end = None, None
            for i, (s, e) in enumerate(tokens["offset_mapping"]):
                if s <= answer_start_char < e:
                    token_start = i
                if s < answer_end_char <= e:
                    token_end = i
            token_start = 0 if token_start is None else token_start
            token_end = 0 if token_end is None else token_end
            start_positions = torch.tensor(token_start)
            end_positions = torch.tensor(token_end)

        return {
            "input_ids": torch.tensor(tokens["input_ids"]),
            "attention_mask": torch.tensor(tokens["attention_mask"]),
            "start_positions": start_positions,
            "end_positions": end_positions
        }

# 3️⃣ Prepare subset datasets
n_train = 10
m_val = 2
languages_to_keep = ['ar','te','ko']

df_train_subset = df_train_correct_bio[df_train_correct_bio['lang'].isin(languages_to_keep)].sample(n=n_train, random_state=42)
df_val_subset   = df_val_correct_bio[df_val_correct_bio['lang'].isin(languages_to_keep)].sample(n=m_val, random_state=42)

train_dataset = QADataset(df_train_subset, tokenizer)
val_dataset   = QADataset(df_val_subset, tokenizer)

# 4️⃣ Simpel EM + F1 funktion
def simple_em_f1(pred_answers, true_answers):
    em, f1 = 0, 0
    for p, t in zip(pred_answers, true_answers):
        p_norm = p.replace(" ", "").lower()
        t_norm = t.replace(" ", "").lower()
        em += int(p_norm == t_norm)

        # F1: token overlap
        p_tokens = p.split()
        t_tokens = t.split()
        common = set(p_tokens) & set(t_tokens)
        if len(common) == 0:
            f1 += 0
        else:
            f1 += 2*len(common)/(len(p_tokens)+len(t_tokens))
    n = len(pred_answers)
    return {"exact_match": 100*em/n, "f1": 100*f1/n}

# 5️⃣ Training arguments
training_args = TrainingArguments(
    output_dir="./distilbert_qa",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=3e-5,
    logging_steps=50,
    save_steps=500,
    weight_decay=0.01,
    save_total_limit=2
)

# 6️⃣ Trainer (uden compute_metrics)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 7️⃣ Train
trainer.train()

# 8️⃣ Predict på valideringssættet
pred_answers = []
true_answers = []
for i in range(len(val_dataset)):
    inputs = {
        "input_ids": val_dataset[i]["input_ids"].unsqueeze(0).to(device),
        "attention_mask": val_dataset[i]["attention_mask"].unsqueeze(0).to(device)
    }
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = outputs.start_logits.argmax()
    end_idx = outputs.end_logits.argmax()
    pred_text = tokenizer.decode(val_dataset[i]["input_ids"][start_idx:end_idx+1], skip_special_tokens=True)
    pred_answers.append(pred_text)
    true_answers.append(df_val_subset.iloc[i]["answer"])

# 9️⃣ Evaluer
results = simple_em_f1(pred_answers, true_answers)
print("Validation results:", results)


Using device: cpu


Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\CoolD\miniconda3\envs\NLP\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


Validation results: {'exact_match': 0.0, 'f1': 0.0}
